# agent_obs_alert

Central alerting notebook for the ISH/SAA agent observability pipeline. This notebook:
1. persists findings to `obs_incidents` (append-only, MERGE-deduped),
2. evaluates each detector condition in Python,
3. bounds every check in time,
4. reports missing/stale upstream tables as `UNAVAILABLE` rather than crashing,
5. posts a triage-oriented Adaptive Card to Teams — once per incident, not once per run,
6. raises when a detector is broken, so Databricks' own job notifications fire.

## 11 active detectors (prod v1)
1. **Handover delivery failures** (CRITICAL) — email send failures from ISH audit
2. **Handover delivery rate** (CRITICAL) — 7-day rolling failure rate >20%, OR failed>=3
   absolute-count floor (added 2026-09-21 — see #4 below) regardless of sample size
3. **Shift context missing** (WARN) — blank shift_date/shift_type/batch_nbr in outputs
4. **Blank output** (CRITICAL) — content is NULL/empty/trivial on an output record
5. **Schema drift / field missing** (CRITICAL) — JSON key appeared or disappeared vs baseline
6. **Capability silence** (WARN) — output_type hasn't generated in >grace_hours
7. **Pipeline heartbeat** (CRITICAL) — zero outputs from any capability in 45 minutes
   (tightened from 2h 2026-09-21 — real max gap over 30 days is 16.4 min)
8. **ETL pipeline health** (CRITICAL) — ETL source data refresh failed or stale >30min
9. **ETL table staleness** (CRITICAL, added 2026-09-21) — a single ETL source table hasn't
   completed a run in >60 min, independent of whether the rest of the fleet is still running.
   Per-table companion to #8's global `etl_pipeline_staleness` scalar — closes a confirmed
   masking gap where a partial ETL stall was invisible to the global check because unaffected
   tables kept its `max(run_timestamp)` fresh. Runs alongside #8, not instead of it.
10. **ETL run slow** (CRITICAL, added 2026-09-18 as WARN; promoted 2026-09-21) — ETL run
    duration degraded beyond its own MAD-based historical bound, clustered (>=3 in an hour),
    rolled up to `(task_name, window_start)`. Promoted after 5 real, correlated, multi-table
    slowdown events were observed in ~1 week while it was WARN/log-only and invisible to a human.
11. **Capability silence ceiling** (CRITICAL, added 2026-09-21 as WARN; promoted CRITICAL and
    tightened 168h→120h 2026-09-21) — backstop for capabilities excluded from #6 because
    `silence_grace_hours IS NULL` (irregular cadence, currently just `sev2-insights`); fires
    only past a much longer fixed ceiling (`silence_ceiling_hours`, 120h for `sev2-insights`).
    Promoted because WARN-forever was itself judged a false-negative risk for a permanent-dark
    event, even though (unlike #10) it has not yet fired on a real occurrence — see
    `threshold_basis` for the historical false-positive-rate analysis behind the 120h number.

## +1 provisional WARN detector
12. **Write lag anomalies** (WARN, added 2026-09-18) — write_lag_s degraded beyond
   `greatest(its own MAD-based historical bound, 300s)` for a capability/scheduler_run,
   clustered (>=3 in an hour). 300s floor added 2026-09-21 — the MAD baseline is degenerate
   (median=MAD=0 everywhere), so the floor stops it from tripping on trivial nonzero lag.
   write_lag_s itself has been exactly 0 for all 10,679 rows over 30 days — whether this
   reflects a genuinely-instant write path or unwired upstream instrumentation is an open
   question, on hold pending the data owner (2026-09-21, FP/FN bias review priority 3).

Ships WARN/observe-only (never persisted to `obs_incidents`, never posted to Teams) because its
threshold is newly introduced/unvalidated against real incident history — see `threshold_basis`,
`status='provisional'`. Promote to CRITICAL only after review.

## Severity semantics
`raise` fires **only on UNAVAILABLE**, so task status means *"is monitoring working"* rather
than *"did it find something"*. CRITICAL findings route to Teams and `obs_incidents`.

## Notification policy
A CRITICAL incident notifies when first detected, then not again for 24 h unless still
unacknowledged. WARN findings are **not** notified. Broken detectors notify every run.

## Thresholds
Recorded with their basis in `mq_gmdf_dev.oil_obs.threshold_basis`.

## Prod source migration (2026-09-10)
Reads from prod source tables via views (`v_llm_bronze` → `mq_gmdf_dp_prd.oil.ptof_primary__ai_shift_outputs`,
`v_ish_bronze` → `mq_gmdf_dp_prd.oil.ptof_ish_audit`, `v_etl_bronze` → `mq_gmdf_dp_prd.oil.ptof_etl_pipeline_audit`).
All writes stay in `mq_gmdf_dev.oil_obs`. 7 detectors dropped (latency, error rate, hallucination,
transport violations, prompt size, credential fastfail, rapid human correction).

In [ ]:
WEBHOOK = dbutils.secrets.get(scope="obs-alerting", key="teams-webhook")
CAT = "mq_gmdf_dev.oil_obs"

print(f"webhook_configured={bool(WEBHOOK)}")

In [ ]:
import json
import re
from datetime import datetime, timezone

results = []  # name, severity, value, detail


def check(name, sql, severity_fn, detail_fn=None):
    """Run a scalar-ish check. Missing table -> UNAVAILABLE (a finding, not a crash)."""
    try:
        rows = spark.sql(sql).collect()
    except Exception as e:  # noqa: BLE001 - any failure should surface as a finding
        results.append({"name": name, "severity": "UNAVAILABLE", "value": None,
                        "detail": str(e).split("\n")[0][:300]})
        return
    if not rows:
        results.append({"name": name, "severity": "OK", "value": 0, "detail": "no rows"})
        return
    row = rows[0].asDict()
    detail = detail_fn(row) if detail_fn else json.dumps(
        {k: (str(v) if v is not None else None) for k, v in row.items()})
    results.append({"name": name, "severity": severity_fn(row), "value": row, "detail": detail})


gt0   = lambda r: "CRITICAL" if (r.get("n") or 0) > 0 else "OK"  # noqa: E731
warn0 = lambda r: "WARN"     if (r.get("n") or 0) > 0 else "OK"  # noqa: E731


## Persist findings to obs_incidents

Runs before the checks so `unacknowledged_critical` sees this run's findings.
MERGE on `(detector, source_row_id)` means re-detecting increments `detection_count` rather
than duplicating, and acknowledgement survives across runs.

Any `SKIPPED` line is a column-name mismatch in `INCIDENT_SOURCES` — fix the list rather
than ignoring it, or that detector's findings are never persisted.

**Re-detection clears `resolved_at` and `acknowledged_at`:** `WHEN MATCHED` sets both to `NULL`
in addition to refreshing `last_detected`/`detection_count`. Without this, a detector whose key
is stable across occurrences (a constant literal id, or `capability`+`field`/`config` with no
per-event discriminator — `blank_output`, `schema_field_missing`, `handover_delivery_rate`,
`pipeline_heartbeat`, `etl_pipeline_staleness`) would go permanently dark the first time it is
resolved-or-acknowledged and later recurs: the notify query, `unacknowledged_critical`, and
auto-resolve itself all filter on both columns being NULL, so a frozen non-null value on either
one silently suppresses every future occurrence with no error. A fresh re-detection is a new
occurrence — an old acknowledgment of a since-resolved instance shouldn't silence it. Detectors
keyed on a genuinely per-event id (`handover_delivery` on `ish_row_id`, `etl_pipeline_failure`
on `run_id`) were never exposed to this, since a resolved occurrence can't recur under the same
id.

**Auto-resolve:** immediately after the MERGE loop, any incident whose detector ran cleanly
this run but did not re-detect it gets `resolved_at` stamped -- the condition stopped
recurring. This is the only place `resolved_at` is set automatically outside of re-detection;
`acknowledged_by`/`acknowledged_at` are otherwise hand-set only (cleared on re-detection above,
never set automatically). A detector that errored this run is excluded from auto-resolve for
its own incidents (see `_skipped_detectors`), so a broken query can never be misread as "the
condition went away."

In [ ]:
# pipeline_heartbeat and etl_pipeline_staleness are scalar CRITICAL checks with no natural
# per-row grain (unlike blank_output/schema_drift/etc, which have a findings table) -- just one
# global condition each. Run them here, before incident persistence, so cell below can MERGE
# their result into obs_incidents this same run instead of only printing to the job log.
#
# Threshold tightened 2h -> 45min (2026-09-21, FP/FN bias review, priority 5, signed off):
# real max gap in v_llm_bronze over 30 days is 16.4 min; 45 min is still 2.7x that (comfortable
# false-positive margin) but cuts the false-negative exposure window by ~63% vs. the prior 2h.
check(
    "pipeline_heartbeat",
    f"""
    SELECT CASE WHEN count(*) = 0 THEN 1 ELSE 0 END AS n, count(*) AS rows_last_45m
    FROM {CAT}.v_llm_bronze
    WHERE called_at >= current_timestamp() - INTERVAL 45 MINUTES
    """,
    gt0,
)

check(
    "etl_pipeline_staleness",
    f"""SELECT CASE WHEN max(run_timestamp) < current_timestamp() - INTERVAL 30 MINUTES
                    THEN 1 ELSE 0 END AS n,
           max(run_timestamp) AS latest_run
        FROM {CAT}.v_etl_bronze""",
    gt0,
)

In [ ]:
INCIDENT_SOURCES = [
    # detector, source table, id column, capability column, severity, payload columns, extra WHERE
    #
    # 6 active detectors that persist to obs_incidents:
    ("handover_delivery",  "handover_delivery_failures", "ish_row_id",
     None,                  "CRITICAL",
     ["failure_reason", "shift_date", "batch_nbr"], ""),
    ("blank_output", "blank_output_findings", "finding_signature",
     "capability", "CRITICAL",
     ["model_config", "blank_rate_window", "blank_count_window", "total_calls_window",
      "latest_hour"], ""),
    ("schema_field_missing", "response_schema_drift", "finding_signature",
     "capability", "CRITICAL",
     ["model_configs_seen", "field_name", "drift_type",
      "baseline_presence_rate", "current_present", "current_rows"],
     "WHERE drift_type = 'field_missing'"),
    ("handover_delivery_rate", "handover_delivery_rate", "'handover_delivery_rate_global'",
     None,                     "CRITICAL",
     ["failure_pct_7d", "sent_ok", "failed", "last_attempt"],
     # Absolute-count OR-trigger added 2026-09-21 (FP/FN bias review, priority 4, signed off):
     # the rate-only path needs (sent_ok + failed) >= 10 to evaluate at all, which structurally
     # can't fire during a low-volume week (current pace ~14 attempts/7d) no matter how bad a
     # small failure cluster is. `failed >= 3` is a hard floor independent of sample size.
     "WHERE (failure_pct_7d > 20 AND (sent_ok + failed) >= 10) OR failed >= 3"),
    ("etl_pipeline_failure", "etl_pipeline_health", "finding_signature",
     None, "CRITICAL",
     ["table_or_view", "status", "error_message", "run_timestamp", "duration_seconds"], ""),
    # etl_table_staleness (added 2026-09-21): per-table companion to the etl_pipeline_staleness
    # scalar below -- table-backed (finding_signature is a constant sha2(table_or_view, 256))
    # so it fits the standard table-backed MERGE loop rather than SCALAR_INCIDENT_SOURCES.
    ("etl_table_staleness", "etl_table_staleness", "finding_signature",
     None, "CRITICAL",
     ["table_or_view", "minutes_since_last_run", "last_run_at"], ""),
    # etl_run_slow (promoted WARN -> CRITICAL 2026-09-21, FP/FN bias review, priority 2, signed
    # off): 5 real, correlated, multi-table slowdown events observed in ~1 week while WARN/
    # log-only -- a proven-real signal that was invisible to a human. finding_signature is
    # sha2(task_name || window_start), so table-backed like the detectors above.
    ("etl_run_slow", "etl_run_slow", "finding_signature",
     None, "CRITICAL",
     ["task_name", "anomalous_count", "affected_table_count", "affected_tables",
      "max_duration_s", "upper_bound_s"], ""),
    # capability_silence_ceiling (promoted WARN -> CRITICAL and tightened 168h -> 120h
    # 2026-09-21, FP/FN bias review follow-up, signed off: "ok lets do it"): backstop for
    # capabilities excluded from capability_silence (WARN, checked in cell 7) because
    # silence_grace_hours IS NULL -- currently just sev2-insights. WARN-forever was judged a
    # false-negative risk in its own right for a permanent-dark event, even though this
    # detector (unlike etl_run_slow) has not yet fired on a real occurrence. finding_signature
    # is a constant sha2(capability, 256), same lifecycle pattern as etl_table_staleness.
    ("capability_silence_ceiling", "capability_silence_ceiling", "finding_signature",
     "capability", "CRITICAL",
     ["silence_ceiling_hours", "hours_since_last_call", "last_call_at", "owner"], ""),
]

# pipeline_heartbeat and etl_pipeline_staleness have no findings table (see the check() calls
# above) -- just a single global scalar condition each. Persisted below as constant-id rows
# straight from that check() result, so they get the same obs_incidents lifecycle (dedup,
# notify/re-notify, auto-resolve) as the table-backed detectors above.
SCALAR_INCIDENT_SOURCES = [
    ("pipeline_heartbeat", "pipeline_heartbeat_global"),
    ("etl_pipeline_staleness", "etl_staleness_global"),
]

# BACKTRACK: "Where to look" queries for the Teams card. Point at the durable bronze views
# (append-only) rather than detector findings tables (CREATE OR REPLACE'd over rolling windows).
# All lineage strings reference prod source tables via the views.
def _sqlq(v):
    return str(v).replace("'", "''") if v is not None else ""

BACKTRACK = {
    "handover_delivery": {
        "table": "v_ish_bronze",
        "where": lambda cap, p, row_id, last_detected: f"id = '{_sqlq(row_id)}'",
        "lineage": "ptof_obs_behavioral_correlation.ipynb -> handover_delivery_failures -> "
                   "v_ish_bronze -> mq_gmdf_dp_prd.oil.ptof_ish_audit",
    },
    "handover_delivery_rate": {
        "table": "v_ish_bronze",
        "lineage": "ptof_obs_behavioral_correlation.ipynb -> handover_delivery_rate -> "
                   "v_ish_bronze -> mq_gmdf_dp_prd.oil.ptof_ish_audit",
        "where": lambda cap, p, row_id, last_detected: (
            "entity_type = 'HandoverEmail' AND ts BETWEEN "
            f"TIMESTAMP'{last_detected}' - INTERVAL 7 DAYS AND TIMESTAMP'{last_detected}'"),
        "order_by": "ts DESC",
    },
    "blank_output": {
        "table": "v_llm_bronze",
        "lineage": "ptof_obs_mal_output.ipynb -> blank_output_findings -> "
                   "v_llm_bronze -> mq_gmdf_dp_prd.oil.ptof_primary__ai_shift_outputs",
        "where": lambda cap, p, row_id, last_detected: (
            f"capability = '{_sqlq(cap)}' AND is_blank_output = true AND called_at BETWEEN "
            f"TIMESTAMP'{last_detected}' - INTERVAL 6 HOURS AND TIMESTAMP'{last_detected}'"),
        "order_by": "called_at DESC",
    },
    "schema_field_missing": {
        "table": "v_llm_bronze",
        "lineage": "ptof_obs_mal_output.ipynb -> response_schema_drift -> "
                   "v_llm_bronze -> mq_gmdf_dp_prd.oil.ptof_primary__ai_shift_outputs",
        "where": lambda cap, p, row_id, last_detected: (
            f"capability = '{_sqlq(cap)}' AND called_at BETWEEN "
            f"TIMESTAMP'{last_detected}' - INTERVAL 24 HOURS AND TIMESTAMP'{last_detected}'"),
        "order_by": "called_at DESC",
        "note": "Field-missing can't be expressed as a plain filter — inspect response_parsed "
                "on these rows directly.",
    },
    "etl_pipeline_failure": {
        "table": "v_etl_bronze",
        "lineage": "ptof_obs_liveness_detection.ipynb -> etl_pipeline_health -> "
                   "v_etl_bronze -> mq_gmdf_dp_prd.oil.ptof_etl_pipeline_audit",
        "where": lambda cap, p, row_id, last_detected: (
            f"table_or_view = '{_sqlq(p.get('table_or_view', ''))}' AND run_timestamp BETWEEN "
            f"TIMESTAMP'{last_detected}' - INTERVAL 24 HOURS AND TIMESTAMP'{last_detected}'"),
        "order_by": "run_timestamp DESC",
    },
    "etl_table_staleness": {
        "table": "v_etl_bronze",
        "lineage": "ptof_obs_liveness_detection.ipynb -> etl_table_staleness -> "
                   "v_etl_bronze -> mq_gmdf_dp_prd.oil.ptof_etl_pipeline_audit",
        "where": lambda cap, p, row_id, last_detected: (
            f"table_or_view = '{_sqlq(p.get('table_or_view', ''))}' AND run_timestamp BETWEEN "
            f"TIMESTAMP'{last_detected}' - INTERVAL 24 HOURS AND TIMESTAMP'{last_detected}'"),
        "order_by": "run_timestamp DESC",
        "note": "Scoped to this one table only -- if etl_pipeline_staleness (the global check) "
                "is quiet, that does NOT mean this is a false alarm; it only fires when EVERY "
                "table stops.",
    },
    "etl_run_slow": {
        "table": "v_etl_bronze",
        "lineage": "ptof_obs_liveness_detection.ipynb -> etl_run_slow -> "
                   "v_etl_bronze -> mq_gmdf_dp_prd.oil.ptof_etl_pipeline_audit",
        "where": lambda cap, p, row_id, last_detected: (
            f"task_name = '{_sqlq(p.get('task_name', ''))}' AND run_timestamp BETWEEN "
            f"TIMESTAMP'{last_detected}' - INTERVAL 24 HOURS AND TIMESTAMP'{last_detected}'"),
        "order_by": "run_timestamp DESC",
        "note": "Rolled up to task_name -- affected_tables in the payload lists which of that "
                "task's tables were slow this occurrence; all of them typically move together.",
    },
    "capability_silence_ceiling": {
        "table": "v_llm_bronze",
        "lineage": "ptof_obs_liveness_detection.ipynb -> capability_silence_ceiling -> "
                   "v_llm_bronze -> mq_gmdf_dp_prd.oil.ptof_primary__ai_shift_outputs",
        "where": lambda cap, p, row_id, last_detected: (
            f"capability = '{_sqlq(cap)}' AND called_at BETWEEN "
            f"TIMESTAMP'{last_detected}' - INTERVAL 30 DAYS AND TIMESTAMP'{last_detected}'"),
        "order_by": "called_at DESC",
        "note": "Expect zero or very sparse rows -- that's the condition. Window widened to "
                "30 days here since this capability's normal cadence is irregular.",
    },
    "pipeline_heartbeat": {
        "table": "v_llm_bronze",
        "lineage": "ptof_obs_alert.ipynb -> pipeline_heartbeat -> "
                   "v_llm_bronze -> mq_gmdf_dp_prd.oil.ptof_primary__ai_shift_outputs",
        "where": lambda cap, p, row_id, last_detected: (
            "called_at BETWEEN "
            f"TIMESTAMP'{last_detected}' - INTERVAL 6 HOURS AND TIMESTAMP'{last_detected}'"),
        "order_by": "called_at DESC",
        "note": "Expect zero rows in the trailing 45min — that's the condition. Window widened "
                "to 6h here to show the most recent activity before it stopped.",
    },
    "etl_pipeline_staleness": {
        "table": "v_etl_bronze",
        "lineage": "ptof_obs_alert.ipynb -> etl_pipeline_staleness -> "
                   "v_etl_bronze -> mq_gmdf_dp_prd.oil.ptof_etl_pipeline_audit",
        "where": lambda cap, p, row_id, last_detected: (
            "run_timestamp BETWEEN "
            f"TIMESTAMP'{last_detected}' - INTERVAL 6 HOURS AND TIMESTAMP'{last_detected}'"),
        "order_by": "run_timestamp DESC",
    },
}

# Timestamp before the emission loop so auto-resolve can distinguish "ran and found nothing"
# from "hasn't run yet". Detectors that error are tracked in _skipped_detectors and excluded
# from auto-resolve.
_incident_run_ts = spark.sql("SELECT current_timestamp() AS ts").first().ts
_skipped_detectors = set()

for detector, table, id_col, cap_col, severity, payload_cols, extra in INCIDENT_SOURCES:
    try:
        cap_expr = cap_col if cap_col else "CAST(NULL AS STRING)"
        payload = ", ".join(f"'{c}', CAST({c} AS STRING)" for c in payload_cols)
        source_expr = f"{CAT}.{table}"
        spark.sql(f"""
            MERGE INTO {CAT}.obs_incidents t
            USING (
              SELECT '{detector}'             AS detector,
                     CAST({id_col} AS STRING) AS source_row_id,
                     {cap_expr}               AS capability,
                     '{severity}'             AS severity,
                     to_json(map({payload}))  AS signal_payload
              FROM {source_expr} {extra}
            ) s
            ON t.detector = s.detector AND t.source_row_id = s.source_row_id
            WHEN MATCHED THEN UPDATE SET
                t.severity        = s.severity,
                t.last_detected   = current_timestamp(),
                t.detection_count = t.detection_count + 1,
                t.signal_payload  = s.signal_payload,
                t.resolved_at     = NULL,
                t.acknowledged_at = NULL,
                t.acknowledged_by = NULL
            WHEN NOT MATCHED THEN INSERT
                (detector, source_row_id, capability, severity,
                 first_detected, last_detected, detection_count, signal_payload)
              VALUES
                (s.detector, s.source_row_id, s.capability, s.severity,
                 current_timestamp(), current_timestamp(), 1, s.signal_payload)
        """)
        print(f"  emitted: {detector}")
    except Exception as e:  # noqa: BLE001
        _skipped_detectors.add(detector)
        print(f"  SKIPPED {detector}: {str(e).split(chr(10))[0][:160]}")

# Scalar checks: MERGE a synthetic constant-id row only when the check() result (computed
# above, before this cell) came back CRITICAL -- skipping the MERGE otherwise has the same
# effect as "no matching rows" for a table-backed detector, so auto-resolve (below) still
# clears a resolved condition without any extra code.
for detector, source_row_id in SCALAR_INCIDENT_SOURCES:
    result = next((r for r in results if r["name"] == detector), None)
    if result is None or result["severity"] == "UNAVAILABLE":
        _skipped_detectors.add(detector)
        print(f"  SKIPPED {detector}: check did not run cleanly")
        continue
    if result["severity"] != "CRITICAL":
        continue
    try:
        payload = json.dumps({k: str(v) for k, v in result["value"].items() if k != "n"})
        spark.sql(f"""
            MERGE INTO {CAT}.obs_incidents t
            USING (
              SELECT '{detector}' AS detector, '{source_row_id}' AS source_row_id,
                     CAST(NULL AS STRING) AS capability, 'CRITICAL' AS severity,
                     '{payload.replace("'", "''")}' AS signal_payload
            ) s
            ON t.detector = s.detector AND t.source_row_id = s.source_row_id
            WHEN MATCHED THEN UPDATE SET
                t.severity        = s.severity,
                t.last_detected   = current_timestamp(),
                t.detection_count = t.detection_count + 1,
                t.signal_payload  = s.signal_payload,
                t.resolved_at     = NULL,
                t.acknowledged_at = NULL,
                t.acknowledged_by = NULL
            WHEN NOT MATCHED THEN INSERT
                (detector, source_row_id, capability, severity,
                 first_detected, last_detected, detection_count, signal_payload)
              VALUES
                (s.detector, s.source_row_id, s.capability, s.severity,
                 current_timestamp(), current_timestamp(), 1, s.signal_payload)
        """)
        print(f"  emitted: {detector}")
    except Exception as e:  # noqa: BLE001
        _skipped_detectors.add(detector)
        print(f"  SKIPPED {detector}: {str(e).split(chr(10))[0][:160]}")

print("Incident emission complete.")

In [ ]:
# Auto-resolve: an incident whose detector ran cleanly this run but didn't re-MERGE it means
# the underlying condition stopped recurring -- last_detected stays behind _incident_run_ts.
# Without this, resolved_at is "set by hand" only, so the open/unacknowledged backlog (235
# CRITICAL incidents observed live, some untouched since first detection) grows unbounded
# regardless of whether the condition is still real. Runs across both acknowledged and
# unacknowledged rows -- resolution reflects the condition's state, not human review status.
#
# Detectors in _skipped_detectors are excluded: a broken query must never be read as "the
# condition went away" (that would silently clear real incidents the moment their detector broke).
_active_detectors = ([d for d, *_ in INCIDENT_SOURCES if d not in _skipped_detectors]
                      + [d for d, _ in SCALAR_INCIDENT_SOURCES if d not in _skipped_detectors])
if _active_detectors:
    _detector_list = ", ".join(f"'{d}'" for d in _active_detectors)
    spark.sql(f"""
        UPDATE {CAT}.obs_incidents
        SET resolved_at = current_timestamp()
        WHERE resolved_at IS NULL
          AND detector IN ({_detector_list})
          AND last_detected < '{_incident_run_ts}'
    """)
    print(f"Auto-resolve checked {len(_active_detectors)} detector(s) that ran cleanly this run.")
else:
    print("No detectors ran cleanly this run; skipping auto-resolve.")

In [ ]:
# === Liveness checks ===

# capability_silence: has each registered capability produced output within its grace window?
# Prod: saa-display (2h), situational-awareness (2h), summary (36h). sev2-insights excluded
# (silence_grace_hours = NULL, irregular cadence).
check(
    "capability_silence",
    f"""
    SELECT count(*) AS n,
           concat_ws(', ', collect_set(concat(capability,'=',
                      cast(hours_since_last_call AS STRING),'h'))) AS quiet
    FROM {CAT}.capability_silence
    """,
    warn0,
)

# capability_silence_ceiling (added 2026-09-21, item #2; promoted WARN -> CRITICAL and
# tightened 168h -> 120h 2026-09-21, FP/FN bias review follow-up, signed off) is no longer
# checked here -- it's table-backed with a finding_signature now and persists via the standard
# INCIDENT_SOURCES MERGE loop above (cell 5), same as etl_table_staleness/etl_run_slow.

# shift_context_missing: blank shift_date/shift_type/batch_nbr in output records.
# WARN, not notified — standing data-quality gap until upstream populates these fields.
check(
    "shift_context_missing",
    f"""
    SELECT count(*) AS n,
           concat_ws(', ', collect_set(concat(capability,'=',
                      cast(total_calls AS STRING),' calls'))) AS affected
    FROM {CAT}.shift_context_missing
    """,
    warn0,
)

# write_lag_anomalies (added 2026-09-18, provisional; 300s floor added 2026-09-21): write_lag_s
# degraded beyond greatest(its own MAD-based historical bound, 300s), clustered (>=3 in an hour).
# WARN, not notified — see threshold_basis for why this ships in observe-only mode.
check(
    "write_lag_anomalies",
    f"""
    SELECT count(*) AS n,
           concat_ws(', ', collect_set(concat(capability,'=',
                      cast(anomalous_count AS STRING),' anomalous in window'))) AS detail
    FROM {CAT}.write_lag_anomalies
    """,
    warn0,
)

# === ETL health checks ===

# etl_pipeline_failure: any ETL task failure in the last 24h. The agent continues producing
# outputs on stale data — capability_silence and pipeline_heartbeat won't fire.
check(
    "etl_pipeline_failure",
    f"""SELECT count(*) AS n,
           concat_ws(', ', collect_set(table_or_view)) AS failed_tables
        FROM {CAT}.etl_pipeline_health""",
    gt0,
)

# etl_table_staleness (added 2026-09-21): per-table companion to etl_pipeline_staleness above --
# catches a partial ETL stall masked by the global max(run_timestamp) staying fresh off
# unaffected tables. See HANDOFF.md for the supplement-vs-replace reasoning.
check(
    "etl_table_staleness",
    f"""SELECT count(*) AS n,
           concat_ws(', ', collect_set(table_or_view)) AS stale_tables
        FROM {CAT}.etl_table_staleness""",
    gt0,
)

# etl_run_slow (added 2026-09-18; grain changed to task_name/window_start 2026-09-21; promoted
# WARN -> CRITICAL 2026-09-21, FP/FN bias review priority 2, signed off): ETL run duration
# degraded beyond its own MAD-based historical bound, clustered (>=3 in an hour) across however
# many tables share that task_name. 5 real, correlated, multi-table slowdown events were
# observed in ~1 week while this was WARN/log-only and invisible to a human -- promoted so it
# now persists to obs_incidents and posts to Teams like the other CRITICALs above.
check(
    "etl_run_slow",
    f"""
    SELECT count(*) AS n,
           concat_ws(', ', collect_set(concat(task_name,'=',
                      cast(anomalous_count AS STRING),' slow runs across ',
                      cast(affected_table_count AS STRING),' table(s)'))) AS detail
    FROM {CAT}.etl_run_slow
    """,
    gt0,
)

# === Output quality checks ===

# blank_output: response content empty on output records. Invisible to every other detector.
check(
    "blank_output",
    f"""
    SELECT count(*) AS n, max(blank_rate_window) AS worst_rate
    FROM {CAT}.blank_output_findings
    """,
    gt0,
)

# schema_drift: any change to a capability's response shape vs its stored baseline.
check(
    "schema_drift",
    f"""
    SELECT count(*) AS n, concat_ws(',', collect_set(capability)) AS capabilities
    FROM {CAT}.response_schema_drift
    WHERE schema_changed = true
    """,
    warn0,
)

# schema_field_missing: a field that was reliably present has disappeared. The real risk is
# downstream code reading a missing field as null/false instead of unknown.
check(
    "schema_field_missing",
    f"""
    SELECT count(*) AS n,
           concat_ws(', ', collect_set(concat(capability,': ',field_name))) AS detail
    FROM {CAT}.response_schema_drift
    WHERE drift_type = 'field_missing'
    """,
    gt0,
)

# nightly_baseline_staleness: if the nightly baseline job fails silently, schema drift runs
# against stale thresholds. 36h gives ~12h grace past the nightly schedule.
check(
    "nightly_baseline_staleness",
    f"""SELECT count(*) AS n FROM {CAT}.response_field_baseline
        WHERE computed_at < current_timestamp() - INTERVAL 36 HOURS""",
    warn0,
)

# === Handover delivery checks ===

# handover_delivery_rate: fires if delivery gets worse than the ~9.3% baseline (rate path,
# needs >=10 attempts to be meaningful) OR an absolute-count floor is hit regardless of sample
# size (added 2026-09-21, FP/FN bias review priority 4, signed off) -- current volume is only
# ~14 attempts/7d, so the rate path alone could never fire on a real failure cluster in a
# low-volume week.
check(
    "handover_delivery_rate",
    f"""
    SELECT failure_pct_7d, sent_ok, failed, last_attempt,
           CASE WHEN (failure_pct_7d > 20 AND (sent_ok + failed) >= 10) OR failed >= 3
                THEN 1 ELSE 0 END AS n
    FROM {CAT}.handover_delivery_rate
    """,
    gt0,
)

# handover_delivery_new_failure: any failure not yet acknowledged. Unlike the rate, this catches
# a single recurrence — each one is a shift handover nobody received.
check(
    "handover_delivery_new_failure",
    f"""
    SELECT count(*) AS n, max(attempted_at) AS latest
    FROM {CAT}.handover_delivery_failures f
    WHERE NOT exists (SELECT 1 FROM {CAT}.obs_incidents i
                      WHERE i.detector = 'handover_delivery'
                        AND i.source_row_id = f.ish_row_id
                        AND i.acknowledged_at IS NOT NULL)
    """,
    gt0,
)

# === Incident state checks ===

# unacknowledged_critical: backstop rollup across every detector. Catches anything flagged
# that nobody has acted on.
check(
    "unacknowledged_critical",
    f"""
    SELECT count(*) AS n,
           concat_ws(', ', collect_set(concat(detector,':',coalesce(capability,'-')))) AS detail,
           min(first_detected) AS oldest
    FROM {CAT}.obs_incidents
    WHERE severity = 'CRITICAL'
      AND acknowledged_at IS NULL
      AND resolved_at IS NULL
    """,
    gt0,
)

# long_running_incident: age-based, not detection_count. Three days unacknowledged is a real signal.
check(
    "long_running_incident",
    f"""
    SELECT count(*) AS n,
           concat_ws(', ', collect_set(concat(detector,' open ',
                      cast(datediff(current_timestamp(), first_detected) AS STRING),'d'))) AS detail
    FROM {CAT}.obs_incidents
    WHERE resolved_at IS NULL
      AND acknowledged_at IS NULL
      AND first_detected <= current_timestamp() - INTERVAL 3 DAYS
    """,
    warn0,
)

In [ ]:
order = {"CRITICAL": 0, "UNAVAILABLE": 1, "WARN": 2, "OK": 3}
results.sort(key=lambda r: order.get(r["severity"], 9))

for r in results:
    print(f"{r['severity']:<12} {r['name']:<40} {r['detail']}")

criticals   = [r for r in results if r["severity"] == "CRITICAL"]
unavailable = [r for r in results if r["severity"] == "UNAVAILABLE"]
warns       = [r for r in results if r["severity"] == "WARN"]


In [ ]:
DETECTOR_META = {
    "handover_delivery": {
        "label": "Shift handover email failed to send",
        "what": "An automated shift handover never reached PFS3_ISH_SME@lists.lilly.com. "
                "The audit row exists; the email does not.",
        "triage": [
            "getaddrinfo failure = DNS resolution to the mail host, not an app bug",
            "Check whether the affected shift was handed over by other means",
            "Baseline is ~9.3% of send attempts since Jun 19 — check handover_delivery_rate to "
            "see whether this is drift or the standing rate",
        ],
    },
    "blank_output": {
        "label": "Agent returned a blank response",
        "what": "An output record exists but its content was empty often enough to exceed the "
                "blank-rate floor. Invisible to every other detector in this system.",
        "triage": [
            "Check whether this is a prompt/template regression or an upstream input problem",
            "blank_rate_window / blank_count_window / total_calls_window show how bad and how big",
            "A blank handover or summary reaching a real person is worse than an error — nobody "
            "else is watching for this",
        ],
    },
    "schema_field_missing": {
        "label": "Expected output field went missing",
        "what": "A field that was reliably present in this capability's response schema has "
                "disappeared. Downstream automation reading a missing field as null/false "
                "instead of unknown is the real risk here.",
        "triage": [
            "Check for a recent prompt-template edit or model swap on this capability",
            "baseline_presence_rate tells you how reliably the field used to appear",
            "All 4 prod capabilities produce grounded narratives with predictable output shapes",
        ],
    },
    "handover_delivery_rate": {
        "label": "Shift handover delivery rate deteriorating",
        "what": "Handover email failures over the trailing 7 days have exceeded the established "
                "baseline, with enough attempts (>=10) for the rate to be meaningful.",
        "triage": [
            "Compare against the ~9.3% historical baseline noted under handover_delivery — "
            "this fires only once it gets worse than that",
            "Check handover_delivery_failures for the underlying getaddrinfo/DNS pattern",
            "sent_ok / failed counts are in the payload below",
        ],
    },
    "etl_pipeline_failure": {
        "label": "ETL source data pipeline failure",
        "what": "A source table refresh in the ETL pipeline failed. The SAA agent "
                "may be running on stale data — outputs look normal but the underlying "
                "alarms, interventions, or cycle status are not current.",
        "triage": [
            "Check table_or_view in the payload — which source table failed?",
            "Check ptof_etl_pipeline_audit for the error_message and duration_seconds",
            "If the ETL is stuck, the agent's outputs are stale but still arriving — "
            "capability_silence and pipeline_heartbeat will NOT fire",
        ],
    },
    "pipeline_heartbeat": {
        "label": "No agent outputs in 45 minutes — total outage",
        "what": "Zero rows have landed in v_llm_bronze across ALL capabilities for 45 minutes "
                "straight. This is the backstop for a total upstream outage -- every other "
                "detector implicitly assumes outputs keep arriving.",
        "triage": [
            "Check whether ptof_primary__ai_shift_outputs is receiving writes at all",
            "rows_last_45m in the payload below confirms the zero",
            "capability_silence (WARN) fires per-capability on a longer grace window; this "
            "fires regardless of capability once ingestion itself stops",
        ],
    },
    "etl_pipeline_staleness": {
        "label": "ETL hasn't completed a run in 30 minutes",
        "what": "The upstream ETL refresh (~19 tables every 10-15 min) has gone quiet for "
                "30+ minutes. Outputs keep arriving (so pipeline_heartbeat/capability_silence "
                "stay quiet) but the agent is reasoning over stale source data.",
        "triage": [
            "latest_run in the payload below is the last completed ETL timestamp",
            "Check whether the ETL job/scheduler itself is stuck or failed to even start a run",
            "etl_pipeline_failure needs an actual failed run recorded -- this fires on the "
            "absence of any run, which that one can't see",
            "etl_table_staleness (per-table) can be firing even when this one is quiet -- this "
            "check only trips when EVERY table stops, not just one",
        ],
    },
    "etl_table_staleness": {
        "label": "ETL source table hasn't run in over an hour",
        "what": "A single ETL source table has gone quiet for 60+ minutes even though other "
                "tables in the fleet are still refreshing on schedule. Per-table companion to "
                "etl_pipeline_staleness -- catches a partial stall that the global check's "
                "fleet-wide max(run_timestamp) would otherwise mask (confirmed gap: the "
                "2026-09-19/20 weekend incident, 14 of 19 tables stalled ~25h while 5 kept the "
                "global max fresh).",
        "triage": [
            "table_or_view in the payload identifies exactly which source stopped",
            "Check whether that table's upstream job/task is stuck, failed silently, or was "
            "descheduled -- last_run_at / minutes_since_last_run show how long it's been out",
            "etl_pipeline_staleness (global) staying quiet does NOT mean this is a false alarm "
            "-- that check only fires when EVERY table stops, this one is scoped to just this "
            "table",
        ],
    },
    "etl_run_slow": {
        "label": "ETL run duration degraded across a task's tables",
        "what": "ETL run duration for this task_name degraded beyond its own MAD-based "
                "historical bound, clustered (>=3 slow runs in an hour). Promoted from WARN "
                "2026-09-21 after 5 real, correlated, multi-table slowdown events were observed "
                "in ~1 week while it was log-only and invisible to a human.",
        "triage": [
            "task_name in the payload identifies the task; affected_tables lists which of its "
            "tables were slow this occurrence -- they typically move together",
            "max_duration_s vs upper_bound_s shows how far past the historical bound this run got",
            "Check whether this correlates with an etl_pipeline_failure or etl_table_staleness "
            "finding around the same window -- a slowdown can precede an outright stall",
        ],
    },
    "capability_silence_ceiling": {
        "label": "Capability has gone silent past its long-cadence ceiling",
        "what": "A capability with an irregular natural cadence (currently only sev2-insights, "
                "structurally excluded from the shorter-grace capability_silence check) hasn't "
                "produced any output in longer than its silence_ceiling_hours backstop. "
                "Promoted from WARN 2026-09-21 -- WARN-forever was itself judged a false-"
                "negative risk for a permanent-dark event, even though this detector has not "
                "yet fired on a real occurrence (unlike etl_run_slow before its promotion).",
        "triage": [
            "hours_since_last_call vs silence_ceiling_hours in the payload shows how far past "
            "the ceiling this is",
            "Check with the capability's owner (in the payload) whether this is a genuine "
            "outage or a legitimate cadence shift that means the ceiling itself needs revisiting",
            "capability_silence (the shorter-grace WARN check) does not cover this capability "
            "at all -- this is the only detection path for it going dark",
        ],
    },
}


def _age(first_detected):
    """Human-readable age."""
    delta = datetime.now(timezone.utc) - first_detected.replace(tzinfo=timezone.utc)
    d, h = delta.days, delta.seconds // 3600
    if d:
        return f"{d}d {h}h old"
    m = (delta.seconds % 3600) // 60
    return f"{h}h {m}m old" if h else f"{m}m old"


def _job_run_url():
    """Link back to this job run. Absent in interactive runs, which is fine."""
    try:
        ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
        host = ctx.tags().get("browserHostName").get()
        job_id = ctx.tags().get("jobId").get()
        run_id = ctx.tags().get("multitaskParentRunId").get()
        return f"https://{host}/jobs/{job_id}/runs/{run_id}"
    except Exception:  # noqa: BLE001
        return None


_EMAIL_LOCAL_PART_RE = re.compile(r"([^,;\s@]+)@")


def _mask_emails(s):
    """Defense-in-depth: mask the local-part of any email before it can reach a Teams card."""
    return _EMAIL_LOCAL_PART_RE.sub("***@", s)


def _facts_from_payload(payload):
    """signal_payload as labelled facts rather than a raw JSON blob."""
    try:
        d = json.loads(payload) if payload else {}
    except Exception:  # noqa: BLE001
        return [{"title": "payload", "value": str(payload)[:200]}]
    pretty = {
        "failure_reason": "Reason", "shift_date": "Shift date", "batch_nbr": "Batch",
        "model_config": "Model config", "model_configs_seen": "Model configs seen",
        "failure_pct_7d": "Failure % (7d)", "sent_ok": "Sent OK", "failed": "Failed",
        "last_attempt": "Last attempt",
        "blank_rate_window": "Blank rate", "blank_count_window": "Blank count",
        "total_calls_window": "Total calls (window)", "latest_hour": "Latest hour",
        "field_name": "Field", "drift_type": "Drift type",
        "baseline_presence_rate": "Baseline presence rate", "current_present": "Currently present",
        "current_rows": "Current rows",
        "table_or_view": "Table/view", "status": "Status", "error_message": "Error message",
        "run_timestamp": "Run timestamp", "duration_seconds": "Duration (s)",
        "rows_last_45m": "Rows last 45m", "latest_run": "Latest ETL run",
        "minutes_since_last_run": "Minutes since last run", "last_run_at": "Last run at",
        "task_name": "Task", "anomalous_count": "Anomalous run count",
        "affected_table_count": "Affected table count", "affected_tables": "Affected tables",
        "max_duration_s": "Max duration (s)", "upper_bound_s": "Upper bound (s)",
        "silence_ceiling_hours": "Silence ceiling (h)",
        "hours_since_last_call": "Hours since last call", "last_call_at": "Last call at",
        "owner": "Owner",
    }
    return [{"title": pretty.get(k, k), "value": _mask_emails(str(v))[:180]}
            for k, v in d.items() if v not in (None, "", "null")]

In [ ]:
def post_teams(incidents, broken):
    """POST a triage-oriented Adaptive Card to the Teams webhook (Workflows-based, expect 202).

    try/except on purpose: a Teams outage must not fail the alert task, or a notification
    problem becomes indistinguishable from a monitoring problem.
    """
    if not WEBHOOK:
        print("No webhook configured; skipping notification.")
        return
    import requests
    import time

    severity = "DETECTOR BROKEN" if broken else "CRITICAL"
    headline = (f"{len(broken)} detector(s) stopped working" if broken
                else f"{len(incidents)} critical finding(s)")

    body = [{
        "type": "Container", "style": "attention", "bleed": True,
        "items": [
            {"type": "TextBlock", "size": "Large", "weight": "Bolder",
             "text": f"{severity} — {headline}"},
            {"type": "TextBlock", "spacing": "None", "isSubtle": True, "wrap": True,
             "text": f"ISH agent observability · "
                     f"{datetime.now(timezone.utc):%Y-%m-%d %H:%M} UTC"},
        ],
    }]

    # Broken detectors first: monitoring has stopped, which outranks anything it found.
    for r in broken:
        body.append({
            "type": "Container", "style": "attention", "separator": True,
            "items": [
                {"type": "TextBlock", "weight": "Bolder", "wrap": True,
                 "text": f"Detector stopped: {r['name']}"},
                {"type": "TextBlock", "wrap": True, "isSubtle": True,
                 "text": "This check could not run — a table is missing or a query threw. "
                         "Nothing is watching this condition until it is fixed."},
                {"type": "TextBlock", "wrap": True, "fontType": "Monospace",
                 "text": r["detail"][:400]},
            ],
        })

    # Group by detector so multiple findings of one kind read as one problem.
    by_detector = {}
    for r in incidents:
        by_detector.setdefault(r["detector"], []).append(r)

    for detector, rows in by_detector.items():
        meta = DETECTOR_META.get(detector, {})
        oldest = min(r["first_detected"] for r in rows)
        caps = sorted({x["capability"] for x in rows if x["capability"]})

        items = [
            {"type": "TextBlock", "size": "Medium", "weight": "Bolder", "wrap": True,
             "text": meta.get("label", detector)},
            {"type": "TextBlock", "spacing": "None", "isSubtle": True, "wrap": True,
             "text": f"`{detector}`"
                     + (f" · {', '.join(caps)}" if caps else "")
                     + f" · {len(rows)} occurrence(s) · oldest {_age(oldest)}"},
        ]
        if meta.get("what"):
            items.append({"type": "TextBlock", "wrap": True, "text": meta["what"]})

        newest = max(rows, key=lambda x: x["first_detected"])
        facts = _facts_from_payload(newest["signal_payload"])
        if facts:
            items.append({"type": "TextBlock", "weight": "Bolder", "size": "Small",
                          "spacing": "Medium",
                          "text": "Most recent occurrence" if len(rows) > 1 else "Details"})
            items.append({"type": "FactSet", "facts": facts})

        # BACKTRACK: a copy-paste query against the durable bronze view, not this detector's
        # own findings table (which is CREATE OR REPLACE'd over a rolling window).
        bt = BACKTRACK.get(detector)
        if bt:
            payload_dict = json.loads(newest["signal_payload"]) if newest["signal_payload"] else {}
            where_clause = bt["where"](newest["capability"], payload_dict,
                                       newest["source_row_id"], newest["last_detected"])
            query = f"SELECT * FROM {CAT}.{bt['table']} WHERE {where_clause}"
            if bt.get("order_by"):
                query += f" ORDER BY {bt['order_by']}"
            query += " LIMIT 20;"
            items.append({"type": "TextBlock", "weight": "Bolder", "size": "Small",
                          "spacing": "Medium", "text": "Where to look"})
            if bt.get("lineage"):
                items.append({"type": "TextBlock", "wrap": True, "spacing": "None",
                              "isSubtle": True, "text": bt["lineage"]})
            if bt.get("note"):
                items.append({"type": "TextBlock", "wrap": True, "spacing": "None",
                              "isSubtle": True, "text": bt["note"]})
            items.append({"type": "TextBlock", "wrap": True, "spacing": "None",
                          "fontType": "Monospace", "text": query})

        if meta.get("triage"):
            items.append({"type": "TextBlock", "weight": "Bolder", "size": "Small",
                          "spacing": "Medium", "text": "Where to start"})
            items += [{"type": "TextBlock", "wrap": True, "spacing": "None", "isSubtle": True,
                       "text": f"• {t}"} for t in meta["triage"]]

        items.append({
            "type": "TextBlock", "wrap": True, "spacing": "Medium", "size": "Small",
            "isSubtle": True, "fontType": "Monospace",
            "text": "Acknowledge (suppresses the alert, keeps the record):<br>"
                    f"UPDATE mq_gmdf_dev.oil_obs.obs_incidents SET "
                    f"acknowledged_by=current_user(), acknowledged_at=current_timestamp() "
                    f"WHERE detector='{detector}' AND acknowledged_at IS NULL;",
        })

        body.append({"type": "Container", "separator": True, "spacing": "Medium",
                     "items": items})

    card = {"type": "AdaptiveCard", "version": "1.4", "body": body,
            "$schema": "http://adaptivecards.io/schemas/adaptive-card.json"}

    run_url = _job_run_url()
    if run_url:
        card["actions"] = [{"type": "Action.OpenUrl", "title": "Open job run", "url": run_url}]

    # One bounded retry: a single transient blip shouldn't cost a durable-failure record.
    for attempt in range(2):
        try:
            resp = requests.post(
                WEBHOOK,
                data=json.dumps({"type": "message", "attachments": [{
                    "contentType": "application/vnd.microsoft.card.adaptive",
                    "content": card}]}),
                headers={"Content-Type": "application/json"}, timeout=30)
            print(f"Teams webhook status={resp.status_code}")
            if resp.status_code >= 400:
                print(f"  response: {resp.text[:300]}")
            break
        except Exception as e:  # noqa: BLE001
            print(f"Teams webhook FAILED (attempt {attempt + 1}/2): {str(e)[:200]}")
            if attempt == 0:
                time.sleep(2)
                continue
            try:
                day_key = datetime.now(timezone.utc).strftime("%Y-%m-%d")
                exc_text = str(e)[:500].replace("'", "''")
                spark.sql(f"""
                    MERGE INTO {CAT}.obs_incidents t
                    USING (
                      SELECT 'alerting_pipeline' AS detector,
                             'teams_notify_failed_{day_key}' AS source_row_id,
                             CAST(NULL AS STRING) AS capability,
                             'CRITICAL' AS severity,
                             to_json(map('error', '{exc_text}')) AS signal_payload
                    ) s
                    ON t.detector = s.detector AND t.source_row_id = s.source_row_id
                    WHEN MATCHED THEN UPDATE SET
                        t.last_detected   = current_timestamp(),
                        t.detection_count = t.detection_count + 1,
                        t.signal_payload  = s.signal_payload,
                        t.resolved_at     = NULL,
                        t.acknowledged_at = NULL,
                        t.acknowledged_by = NULL
                    WHEN NOT MATCHED THEN INSERT
                        (detector, source_row_id, capability, severity,
                         first_detected, last_detected, detection_count, signal_payload)
                      VALUES
                        (s.detector, s.source_row_id, s.capability, s.severity,
                         current_timestamp(), current_timestamp(), 1, s.signal_payload)
                """)
            except Exception as merge_e:  # noqa: BLE001
                print(f"  also failed to record durable failure: {str(merge_e)[:200]}")

In [ ]:
# Notify once per incident, not once per run.
#
# Without this, a persistent finding posts every 5 minutes forever — the alert-fatigue failure this
# design exists to prevent. obs_incidents.notified_at carries the state: an incident is reported
# when first seen, then not again for 24 h unless it is still unacknowledged.
#
# WARN findings are deliberately NOT notified. shift_context_missing and
# hallucination_signal_liveness are standing conditions; posting them would drown the channel.
# They stay visible in the task output above and in obs_incidents.
try:
    to_notify = spark.sql(f"""
        SELECT detector, source_row_id, capability, severity, first_detected, last_detected,
               detection_count, signal_payload
        FROM {CAT}.obs_incidents
        WHERE severity = 'CRITICAL'
          AND acknowledged_at IS NULL
          AND resolved_at IS NULL
          AND (notified_at IS NULL
               OR notified_at < current_timestamp() - INTERVAL 24 HOURS)
        ORDER BY first_detected
    """).collect()
except Exception as e:  # noqa: BLE001
    print(f"incident notify query failed: {str(e).split(chr(10))[0][:160]}")
    to_notify = []

if to_notify or unavailable:
    post_teams(to_notify, unavailable)
    if to_notify:
        spark.sql(f"""
            UPDATE {CAT}.obs_incidents
            SET notified_at = current_timestamp()
            WHERE severity = 'CRITICAL'
              AND acknowledged_at IS NULL AND resolved_at IS NULL
              AND (notified_at IS NULL
                   OR notified_at < current_timestamp() - INTERVAL 24 HOURS)
        """)
else:
    print("No new or stale incidents to notify.")


In [ ]:
# Raise ONLY on UNAVAILABLE. Task status means "is monitoring working", not "did it find
# something".
#
# UNAVAILABLE means a table is missing or a query threw: a detector silently stopped monitoring,
# which is the condition that hid every problem in this build. That is worth failing a task over.
# CRITICAL findings are tracked in obs_incidents with acknowledgement state and routed to Teams.
if unavailable:
    raise Exception("ISH observability — DETECTOR BROKEN: " + "; ".join(
        f"{r['name']}={r['severity']}" for r in unavailable))

if criticals:
    print(f"\n{len(criticals)} CRITICAL finding(s) above. Task not failed: findings route via "
          f"obs_incidents and Teams, not task status.")
else:
    print("\nNo CRITICAL findings.")